<a href="https://colab.research.google.com/github/NAITOUAHMANEAbdellah/LLM-TP/blob/main/TP_RAG_Altea_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP RAG en entreprise – Altéa Services (Colab, zéro installation locale)

**How to run:** `Runtime > Change runtime type > T4 GPU`, then `Runtime > Run all`. Everything (fake data, indexing, search, LLM, web UI) runs online.
The first run downloads two models (~3 GB), so allow a few minutes.

Pipeline: 1 Collecte → 2 Nettoyage → 3 Découpage → 4 Indexation → 5 Recherche → 6 Génération → 7 Évaluation → 8 Interface

## 0. Installation (une seule cellule)

In [2]:
!pip -q install pypdf chromadb sentence-transformers reportlab gradio

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.
betterproto 2.0.0b7 requires python-dateutil<3.0,>=2.8, but you have python-dateutil 4.0.0 which is incompatible.
holidays 0.107 requires python-dateutil<3,>=2.9.0.post0, but you have python-dateutil 4.0.0 which is incompatible.
ibis-framework 9.5.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
bigframes 2.51.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
google-cloud-bigquery 3.46.2 requires python-dateutil<3.0.0,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires op

## Données aléatoires : 15 documents fictifs (RH / IT / General)
Les valeurs (jours de congés, etc.) sont tirées au hasard : on connaît donc la bonne réponse pour l'évaluation.

In [3]:
import random, textwrap, pathlib, shutil
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import A4

random.seed()            # change à chaque exécution -> données différentes
F = dict(
    conges=random.choice([25, 26, 28, 30]),
    tele=random.choice([2, 3]),
    repas=random.choice([18, 20, 22]),
    mut=random.choice([50, 60, 70]),
    form=random.choice([3, 4, 5]),
    mdp=random.choice([12, 14, 16]),
    vpn=f"vpn{random.randint(1,9)}.altea-services.fr",
    delai=random.choice([5, 7, 10]),
    phish=f"phishing{random.randint(1,9)}@altea-services.fr",
    etage=random.choice([2, 3, 4]),
    ouv=random.choice(["8h30", "9h00"]),
    velo=random.choice([50, 60]),
)

DOCS = {
 "conges_absences": ("RH", "pdf", f"Chaque salarié à temps plein a droit à {F['conges']} jours de congés payés par an. Les congés se posent dans l'outil RH au moins 15 jours à l'avance et doivent être validés par le manager. Les jours non pris au 31 mai sont perdus sauf accord écrit des RH. En cas de maladie, le salarié prévient son manager avant 10h et envoie son arrêt de travail sous 48 heures. Les congés pour événements familiaux (mariage, naissance, décès) sont accordés sur justificatif."),
 "teletravail": ("RH", "pdf", f"Le télétravail est autorisé {F['tele']} jours par semaine après validation du manager. Les jours de présence obligatoire au bureau sont fixés par chaque équipe. Le salarié doit être joignable pendant les horaires habituels et utiliser le VPN de l'entreprise. Une indemnité de 10 euros par mois est versée pour les frais liés au télétravail. Le télétravail est suspendu pendant la période d'essai."),
 "notes_de_frais": ("RH", "txt", f"Les frais professionnels sont remboursés sur présentation des justificatifs dans l'outil de notes de frais, avant le 5 du mois suivant. Le plafond pour un repas en déplacement est de {F['repas']} euros. Le train en seconde classe est la règle par défaut ; l'avion nécessite l'accord du directeur. Les notes sont remboursées par virement sous 10 jours après validation."),
 "mutuelle": ("RH", "pdf", f"La mutuelle d'entreprise est obligatoire dès le premier jour du contrat. L'entreprise prend en charge {F['mut']} % de la cotisation du salarié. Les ayants droit (conjoint, enfants) peuvent être ajoutés moyennant une cotisation supplémentaire. La carte de tiers payant est envoyée sous 3 semaines. Une dispense est possible si le salarié est déjà couvert par la mutuelle de son conjoint, sur justificatif."),
 "formation": ("RH", "txt", f"Chaque salarié dispose de {F['form']} jours de formation par an. La demande se fait auprès des RH après l'entretien annuel. Les formations doivent être en lien avec le poste. Un bilan de la formation est partagé avec le manager dans le mois qui suit. Les formations en ligne sont accessibles depuis la plateforme interne."),
 "mot_de_passe_oublie": ("IT", "txt", f"Si vous avez oublié votre mot de passe, rendez-vous sur le portail de réinitialisation et cliquez sur Mot de passe oublié. Un lien est envoyé sur votre adresse professionnelle et reste valable 30 minutes. Le nouveau mot de passe doit comporter au moins {F['mdp']} caractères, dont un chiffre et un symbole. Après 5 tentatives échouées, le compte est bloqué : contactez le support IT au poste 4400. Le mot de passe doit être changé tous les 6 mois."),
 "vpn": ("IT", "pdf", f"Pour se connecter au VPN, installez le client fourni par le service IT puis saisissez l'adresse {F['vpn']}. Authentifiez-vous avec votre compte professionnel et le code à usage unique de l'application mobile. Le VPN est obligatoire pour accéder aux outils internes hors du bureau. En cas d'échec de connexion, vérifiez votre accès internet puis ouvrez un ticket au support."),
 "demande_materiel": ("IT", "pdf", f"Toute demande de matériel (ordinateur, écran, casque) se fait via le formulaire du portail IT, avec l'accord du manager. Le délai de livraison est de {F['delai']} jours ouvrés. Un ordinateur portable est renouvelé tous les 4 ans. Le matériel doit être restitué au service IT lors du départ du salarié. En cas de perte ou de vol, prévenir le support IT sous 24 heures."),
 "securite_emails": ("IT", "txt", f"Ne cliquez jamais sur un lien d'un e-mail suspect et n'ouvrez pas les pièces jointes inattendues. Pour signaler un e-mail de phishing, transférez-le à {F['phish']}. Aucun service de l'entreprise ne vous demandera votre mot de passe par e-mail. Vérifiez toujours l'adresse de l'expéditeur. Les e-mails confidentiels doivent être chiffrés."),
 "imprimantes": ("IT", "txt", "Les imprimantes se trouvent à chaque étage près de la salle de pause. Pour imprimer, passez votre badge sur l'appareil et choisissez le document dans la file d'attente. Les impressions non récupérées sont supprimées au bout de 24 heures. L'impression couleur est réservée aux documents destinés aux clients. En cas de bourrage papier, contactez le support IT."),
 "reglement_interieur": ("General", "pdf", "Le présent règlement s'applique à tous les salariés. Chacun doit respecter les horaires, les consignes de sécurité et la confidentialité des informations. Il est interdit de fumer dans les locaux. Les visiteurs doivent être accompagnés et porter un badge visiteur. Tout manquement peut donner lieu à une sanction prévue par le code du travail. Le harcèlement n'est pas toléré et peut être signalé aux RH."),
 "plan_acces": ("General", "pdf", f"Le siège est situé au 12 rue de l'Exemple, à Paris. L'accueil se trouve au {F['etage']}e étage. Le site est accessible par la ligne de métro 9, station Exemple, à 5 minutes à pied. Les visiteurs s'annoncent à l'accueil. L'accès pour les personnes à mobilité réduite se fait par l'entrée latérale côté cour."),
 "horaires": ("General", "txt", f"Les bureaux sont ouverts du lundi au vendredi de {F['ouv']} à 18h00. Le vendredi, la fermeture est à 17h00. Les salariés peuvent arriver entre 8h et 10h dans le cadre des horaires flexibles. La pause déjeuner dure au minimum 45 minutes. Le site est fermé les jours fériés."),
 "parking_velos": ("General", "txt", f"Le parking vélos est situé au sous-sol et accessible avec le badge. L'entreprise rembourse {F['velo']} % de l'abonnement de transports en commun. Les places de parking voiture sont réservées aux personnes à mobilité réduite et aux covoitureurs. Un forfait mobilité durable est proposé aux salariés venant à vélo."),
 "restaurant": ("General", "txt", "Le restaurant d'entreprise est ouvert de 12h à 14h. Les repas sont facturés avec une participation de l'entreprise. Le menu de la semaine est affiché à l'entrée et sur l'intranet. Les salariés peuvent apporter leur repas et le réchauffer dans l'espace cuisine. Un menu végétarien est proposé chaque jour."),
}

def ecrire_pdf(chemin, texte):
    lignes = textwrap.wrap(texte, 85)
    pages = [lignes[i:i + 5] for i in range(0, len(lignes), 5)]
    c = canvas.Canvas(str(chemin), pagesize=A4)
    for n, bloc in enumerate(pages, 1):
        c.setFont("Helvetica", 9);  c.drawString(50, 800, "Altéa Services - Document interne")
        c.setFont("Helvetica", 11)
        y = 760
        for l in bloc:
            c.drawString(50, y, l); y -= 16
        c.setFont("Helvetica", 9);  c.drawString(50, 40, f"Page {n} / {len(pages)}")
        c.showPage()
    c.save()

shutil.rmtree("docs", ignore_errors=True)
for nom, (service, ext, texte) in DOCS.items():
    dossier = pathlib.Path("docs") / service
    dossier.mkdir(parents=True, exist_ok=True)
    if ext == "pdf":
        ecrire_pdf(dossier / f"{nom}.pdf", texte)
    else:
        (dossier / f"{nom}.txt").write_text(
            f"Altéa Services - Document interne\n\n{texte}\n\nPage 1 / 1", encoding="utf-8")
print(len(DOCS), "documents créés dans docs/")
print(F)

15 documents créés dans docs/
{'conges': 26, 'tele': 2, 'repas': 20, 'mut': 50, 'form': 4, 'mdp': 14, 'vpn': 'vpn1.altea-services.fr', 'delai': 5, 'phish': 'phishing2@altea-services.fr', 'etage': 2, 'ouv': '8h30', 'velo': 50}


## Étape 1 – Collecte
On lit chaque fichier et on garde, par page : texte, fichier, page, service. **(Q1 : nombre de pages par document)**

In [4]:
from pypdf import PdfReader
from collections import Counter

def collecter(dossier="docs"):
    pages = []
    for f in sorted(pathlib.Path(dossier).rglob("*")):
        service = f.parent.name
        if f.suffix == ".pdf":
            for i, p in enumerate(PdfReader(f).pages, 1):
                pages.append({"texte": p.extract_text() or "", "source": f.name, "page": i, "service": service})
        elif f.suffix == ".txt":
            pages.append({"texte": f.read_text(encoding="utf-8"), "source": f.name, "page": 1, "service": service})
    return pages

pages = collecter()
for src, n in Counter(p["source"] for p in pages).items():
    print(f"{src:28s} {n} page(s)")
print("TOTAL :", len(pages), "pages")

horaires.txt                 1 page(s)
parking_velos.txt            1 page(s)
plan_acces.pdf               1 page(s)
reglement_interieur.pdf      1 page(s)
restaurant.txt               1 page(s)
demande_materiel.pdf         1 page(s)
imprimantes.txt              1 page(s)
mot_de_passe_oublie.txt      1 page(s)
securite_emails.txt          1 page(s)
vpn.pdf                      1 page(s)
conges_absences.pdf          2 page(s)
formation.txt                1 page(s)
mutuelle.pdf                 1 page(s)
notes_de_frais.txt           1 page(s)
teletravail.pdf              1 page(s)
TOTAL : 16 pages


## Étape 2 – Nettoyage
Espaces, lignes vides, mots coupés, **en-tête répété et numéro de page** (Q4).

In [5]:
import re

def nettoyer(t):
    t = re.sub(r"-\n(\w)", r"\1", t)                         # mots coupés en fin de ligne
    t = re.sub(r"^(Altéa Services.*|Page \d+ ?/ ?\d+)$", "", t, flags=re.M)   # en-tête + n° de page
    t = re.sub(r"[ \t]+", " ", t)                            # espaces multiples
    t = re.sub(r"\n\s*\n+", "\n", t)                         # lignes vides
    return t.strip()

print("AVANT :\n", pages[0]["texte"][:300], "\n")
avant = len(pages)
for p in pages:
    p["texte"] = nettoyer(p["texte"])
pages = [p for p in pages if p["texte"]]
print("APRÈS :\n", pages[0]["texte"][:300])
print(f"\nPages : {avant} avant, {len(pages)} après suppression des pages vides")

AVANT :
 Altéa Services - Document interne

Les bureaux sont ouverts du lundi au vendredi de 8h30 à 18h00. Le vendredi, la fermeture est à 17h00. Les salariés peuvent arriver entre 8h et 10h dans le cadre des horaires flexibles. La pause déjeuner dure au minimum 45 minutes. Le site est fermé les jours fériés 

APRÈS :
 Les bureaux sont ouverts du lundi au vendredi de 8h30 à 18h00. Le vendredi, la fermeture est à 17h00. Les salariés peuvent arriver entre 8h et 10h dans le cadre des horaires flexibles. La pause déjeuner dure au minimum 45 minutes. Le site est fermé les jours fériés.

Pages : 16 avant, 16 après suppression des pages vides


## Étape 3 – Découpage (chunking)
Test de contrôle (Q5) puis tableau des 3 tailles (Q6).

In [6]:
def decouper(texte, taille=500, chevauchement=50):
    pas = taille - chevauchement
    chunks = []
    for debut in range(0, len(texte), pas):
        chunks.append(texte[debut:debut + taille])
        if debut + taille >= len(texte):
            break
    return chunks

assert decouper("abcdefghij", 4, 1) == ["abcd", "defg", "ghij"]
print("Test Q5 OK")

import pandas as pd
lignes = []
for taille, chev in [(200, 20), (500, 50), (1000, 100)]:
    tous = [c for p in pages for c in decouper(p["texte"], taille, chev)]
    lignes.append({"Taille": taille, "Chevauchement": chev, "Nb chunks": len(tous),
                   "Longueur moyenne": round(sum(map(len, tous)) / len(tous))})
pd.DataFrame(lignes)

Test Q5 OK


,Taille,Chevauchement,Nb chunks,Longueur moyenne
0,200,20,37,156
1,500,50,16,335
2,1000,100,16,335


## Étape 4 – Embeddings et indexation
Taille retenue : **500 / 50**. Dimension du vecteur (Q8), matrice de similarité (Q9), index avec `upsert` (Q10, Q12).

In [7]:
from sentence_transformers import SentenceTransformer
import chromadb

modele = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
print("Dimension d'un vecteur :", modele.get_sentence_embedding_dimension())

# Q9 : matrice de similarité
phrases = ["Comment réinitialiser mon mot de passe ?",
           "J'ai perdu mes identifiants de connexion.",
           "Quel est le menu du restaurant ?",
           "Combien de jours de congés ai-je ?"]
from sentence_transformers import util
print(pd.DataFrame(util.cos_sim(modele.encode(phrases), modele.encode(phrases)).numpy().round(2)))

# Indexation
TAILLE, CHEV = 500, 50
client = chromadb.PersistentClient("base_vectorielle")
col = client.get_or_create_collection("altea", metadata={"hnsw:space": "cosine"})

ids, textes, metas = [], [], []
for p in pages:
    for n, ch in enumerate(decouper(p["texte"], TAILLE, CHEV)):
        ids.append(f'{p["source"]}-p{p["page"]}-c{n}')
        textes.append(ch)
        metas.append({"source": p["source"], "page": p["page"], "service": p["service"]})

col.upsert(ids=ids, documents=textes, metadatas=metas,
           embeddings=modele.encode(textes).tolist())
print("Chunks produits :", len(ids), "| indexés :", col.count())
assert col.count() == len(ids)
print("Nombres stockés :", len(ids), "x 384 =", len(ids) * 384)

col.upsert(ids=ids, documents=textes, metadatas=metas, embeddings=modele.encode(textes).tolist())
print("Après 2e upsert (Q12) :", col.count(), "-> pas de doublon")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipykernel_2701/3850988483.py:5: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Dimension d'un vecteur :", modele.get_sentence_embedding_dimension())


Dimension d'un vecteur : 384
      0     1     2     3
0  1.00  0.36  0.01  0.08
1  0.36  1.00  0.05  0.05
2  0.01  0.05  1.00  0.12
3  0.08  0.05  0.12  1.00
Chunks produits : 16 | indexés : 16
Nombres stockés : 16 x 384 = 6144
Après 2e upsert (Q12) : 16 -> pas de doublon


## Étape 5 – Recherche
`rechercher` avec score, test de k (Q2) et filtre par service (Q3).

In [8]:
def rechercher(question, k=3, service=None):
    v = modele.encode([question]).tolist()
    r = col.query(query_embeddings=v, n_results=k, where={"service": service} if service else None)
    return [{"texte": t, "score": round(1 - d, 3), **m}
            for t, d, m in zip(r["documents"][0], r["distances"][0], r["metadatas"][0])]

for q in ["Combien de jours de télétravail ?", "Comment me connecter au VPN ?", "Quel est le plafond d'un repas en déplacement ?"]:
    print("\n>>", q)
    for c in rechercher(q, k=3):
        print(f'   {c["score"]:.3f}  {c["source"]} p.{c["page"]} ({c["service"]})')

print("\n--- Q2 : effet de k ---")
for k in (1, 3, 10):
    print(k, [c["source"] for c in rechercher("Combien de jours de congés ?", k=k)])

print("\n--- Q3 : filtre par service ---")
for s in ("RH", "IT"):
    print(s, [(c["source"], c["score"]) for c in rechercher("Combien de jours de congés ?", k=3, service=s)])


>> Combien de jours de télétravail ?
   0.603  teletravail.pdf p.1 (RH)
   0.500  horaires.txt p.1 (General)
   0.483  conges_absences.pdf p.1 (RH)

>> Comment me connecter au VPN ?
   0.841  vpn.pdf p.1 (IT)
   0.338  securite_emails.txt p.1 (IT)
   0.284  teletravail.pdf p.1 (RH)

>> Quel est le plafond d'un repas en déplacement ?
   0.513  restaurant.txt p.1 (General)
   0.331  notes_de_frais.txt p.1 (RH)
   0.241  horaires.txt p.1 (General)

--- Q2 : effet de k ---
1 ['conges_absences.pdf']
3 ['conges_absences.pdf', 'horaires.txt', 'teletravail.pdf']
10 ['conges_absences.pdf', 'horaires.txt', 'teletravail.pdf', 'formation.txt', 'demande_materiel.pdf', 'notes_de_frais.txt', 'restaurant.txt', 'mutuelle.pdf', 'mot_de_passe_oublie.txt', 'plan_acces.pdf']

--- Q3 : filtre par service ---
RH [('conges_absences.pdf', 0.657), ('teletravail.pdf', 0.476), ('formation.txt', 0.448)]
IT [('demande_materiel.pdf', 0.403), ('mot_de_passe_oublie.txt', 0.272), ('imprimantes.txt', 0.243)]


## Étape 6 – Génération
LLM open source qui tourne dans Colab (Qwen2.5-1.5B-Instruct, pas de clé API). `do_sample=False` = `temperature=0` (Q7).

In [9]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

NOM_LLM = "Qwen/Qwen2.5-1.5B-Instruct"       # sans GPU : "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(NOM_LLM)
llm = AutoModelForCausalLM.from_pretrained(NOM_LLM, torch_dtype=torch.float16, device_map="auto")

def generer(prompt):
    texte = tok.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True)
    entree = tok(texte, return_tensors="pt").to(llm.device)
    sortie = llm.generate(**entree, max_new_tokens=200, do_sample=False)      # déterministe
    return tok.decode(sortie[0][entree["input_ids"].shape[1]:], skip_special_tokens=True).strip()

REPLI = "Je n'ai pas trouvé cette information dans la documentation interne."
CONSIGNE = ("Tu es l'assistant interne d'Altéa Services. Réponds UNIQUEMENT à partir des extraits fournis, "
            "en français, de façon brève. Cite la source sous la forme [fichier, p.X]. "
            f"Si la réponse n'est pas dans les extraits, réponds exactement : {REPLI}")

def construire_prompt(question, chunks):
    extraits = "\n\n".join(f'[{c["source"]}, p.{c["page"]}] {c["texte"]}' for c in chunks)
    return f"{CONSIGNE}\n\nEXTRAITS :\n{extraits}\n\nQUESTION : {question}\nRÉPONSE :"

def repondre(question, k=3, service=None):
    chunks = rechercher(question, k, service)
    return generer(construire_prompt(question, chunks)), chunks

# Q4 : voir le prompt
print(construire_prompt("Combien de jours de télétravail ?", rechercher("Combien de jours de télétravail ?")))

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Tu es l'assistant interne d'Altéa Services. Réponds UNIQUEMENT à partir des extraits fournis, en français, de façon brève. Cite la source sous la forme [fichier, p.X]. Si la réponse n'est pas dans les extraits, réponds exactement : Je n'ai pas trouvé cette information dans la documentation interne.

EXTRAITS :
[teletravail.pdf, p.1] Le télétravail est autorisé 2 jours par semaine après validation du manager. Les
jours de présence obligatoire au bureau sont fixés par chaque équipe. Le salarié doit
être joignable pendant les horaires habituels et utiliser le VPN de l'entreprise. Une
indemnité de 10 euros par mois est versée pour les frais liés au télétravail. Le
télétravail est suspendu pendant la période d'essai.

[horaires.txt, p.1] Les bureaux sont ouverts du lundi au vendredi de 8h30 à 18h00. Le vendredi, la fermeture est à 17h00. Les salariés peuvent arriver entre 8h et 10h dans le cadre des horaires flexibles. La pause déjeuner dure au minimum 45 minutes. Le site est fermé les jour

In [10]:
# Q5 : 3 questions avec réponse + 2 sans réponse
for q in ["Combien de jours de congés payés par an ?", "Quelle est l'adresse du VPN ?",
          "Quelle est la prise en charge de la mutuelle ?",
          "Quel temps fera-t-il demain ?", "Quelle est la capitale de l'Australie ?"]:
    rep, _ = repondre(q)
    print(f"\nQ : {q}\nR : {rep}")

# Q6 : LLM seul (sans extraits) vs RAG
q = "Combien de jours de télétravail par semaine sont autorisés chez Altéa ?"
print("\n--- LLM SEUL ---\n", generer(q))
print("--- RAG ---\n", repondre(q)[0], f"\n(vérité : {F['tele']} jours)")


Q : Combien de jours de congés payés par an ?
R : Chaque salarié à temps plein a droit à 26 jours de congés payés par an.

Q : Quelle est l'adresse du VPN ?
R : L'adresse du VPN est vpn1.altea-services.fr.

Q : Quelle est la prise en charge de la mutuelle ?
R : La mutuelle d'entreprise prend en charge 50% de la cotisation du salarié. Les ayants droit peuvent être ajoutés avec une cotisation supplémentaire.

Q : Quel temps fera-t-il demain ?
R : Je n'ai pas trouvé cette information dans la documentation interne.

Q : Quelle est la capitale de l'Australie ?
R : Je n'ai pas trouvé cette information dans la documentation interne.

--- LLM SEUL ---
 Je suis désolé, mais je n'ai pas d'informations spécifiques sur le nombre exact de jours de télétravail autorisés chez Altéa. Cela pourrait varier en fonction des réglementations locales et des politiques individuelles de l'entreprise.

Pour obtenir une réponse précise, vous devriez contacter directement Altéa ou consulter leur site web officie

## Étape 7 – Évaluation
Grâce aux données aléatoires, on connaît la bonne réponse : la note de justesse est automatique (Q8 à Q10).

In [11]:
EVAL = [
 ("Combien de jours de congés payés par an ?",           "conges_absences.pdf",     str(F["conges"])),
 ("Combien de jours de télétravail par semaine ?",       "teletravail.pdf",         str(F["tele"])),
 ("Quel est le plafond d'un repas en déplacement ?",     "notes_de_frais.txt",      str(F["repas"])),
 ("Quel pourcentage de la mutuelle l'entreprise paie-t-elle ?", "mutuelle.pdf",     str(F["mut"])),
 ("Quelle longueur minimale pour le mot de passe ?",     "mot_de_passe_oublie.txt", str(F["mdp"])),
 ("Quelle est l'adresse du VPN ?",                       "vpn.pdf",                 F["vpn"]),
 ("Quel est le délai de livraison du matériel ?",        "demande_materiel.pdf",    str(F["delai"])),
 ("À quelle adresse signaler un e-mail de phishing ?",   "securite_emails.txt",     F["phish"]),
 ("À quelle heure ouvrent les bureaux ?",                "horaires.txt",            F["ouv"]),
 ("À quel étage se trouve l'accueil ?",                  "plan_acces.pdf",          str(F["etage"])),
]
df = pd.DataFrame(EVAL, columns=["question", "document_attendu", "reponse_attendue"])
df[["question", "document_attendu"]].to_csv("evaluation.csv", index=False)      # Q8

def evaluer(k=3):
    lignes = []
    for q, doc, attendu in EVAL:
        rep, chunks = repondre(q, k=k)
        lignes.append({"question": q,
                       "bon_doc_trouve": doc in [c["source"] for c in chunks],       # Q9 : recherche
                       "reponse_juste": attendu.lower() in rep.lower(),               # Q10 : justesse
                       "bien_source": doc.rsplit(".", 1)[0] in rep,                   # Q10 : source citée
                       "reponse": rep})
    return pd.DataFrame(lignes)

res = evaluer(k=3)
print(f"Recherche   : {res.bon_doc_trouve.mean():.0%}")
print(f"Réponses justes   : {res.reponse_juste.mean():.0%}")
print(f"Bien sourcées     : {res.bien_source.mean():.0%}")
res[["question", "bon_doc_trouve", "reponse_juste", "bien_source"]]

Recherche   : 100%
Réponses justes   : 100%
Bien sourcées     : 20%


,question,bon_doc_trouve,reponse_juste,bien_source
0,Combien de jours de congés payés par an ?,True,True,False
1,Combien de jours de télétravail par semaine ?,True,True,False
2,Quel est le plafond d'un repas en déplacement ?,True,True,False
3,Quel pourcentage de la mutuelle l'entreprise p...,True,True,True
4,Quelle longueur minimale pour le mot de passe ?,True,True,False
5,Quelle est l'adresse du VPN ?,True,True,True
6,Quel est le délai de livraison du matériel ?,True,True,False
7,À quelle adresse signaler un e-mail de phishing ?,True,True,False
8,À quelle heure ouvrent les bureaux ?,True,True,False
9,À quel étage se trouve l'accueil ?,True,True,False


In [12]:
# Q11 : diagnostiquer une erreur, puis tester une amélioration (k = 5)
fausses = res[~res.reponse_juste]
if len(fausses):
    f = fausses.iloc[0]
    print("Réponse fausse :", f.question, "\nBon document trouvé ?", f.bon_doc_trouve,
          "->", "erreur de GÉNÉRATION" if f.bon_doc_trouve else "erreur de RECHERCHE")
res5 = evaluer(k=5)
print(f"\nAvant (k=3) : justes {res.reponse_juste.mean():.0%} | recherche {res.bon_doc_trouve.mean():.0%}")
print(f"Après (k=5) : justes {res5.reponse_juste.mean():.0%} | recherche {res5.bon_doc_trouve.mean():.0%}")


Avant (k=3) : justes 100% | recherche 100%
Après (k=5) : justes 100% | recherche 100%


## Étape 8 – Interface web
Gradio génère un lien public temporaire (`share=True`) : cliquez sur le lien *.gradio.live* et faites la capture d'écran (Q12).

In [13]:
import gradio as gr

def interface(question, service):
    rep, chunks = repondre(question, k=3, service=None if service == "Tous" else service)
    sources = "\n".join(f'- {c["source"]}, p.{c["page"]} (score {c["score"]})' for c in chunks)
    return rep, sources

demo = gr.Interface(fn=interface,
                    inputs=[gr.Textbox(label="Votre question"), gr.Dropdown(["Tous", "RH", "IT", "General"], value="Tous", label="Service")],
                    outputs=[gr.Textbox(label="Réponse"), gr.Textbox(label="Sources")],
                    title="Assistant documentaire Altéa")
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ebed004b5ef4d64f7d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


*texte en italique*## GitHub (Q13)
Easiest way: in Colab, `File > Save a copy in GitHub`, pick/create a repository, done.
Add a `README.md` (goal, install, order of cells, example question, step-7 results, limits, authors) and a `.gitignore` containing `base_vectorielle/`.